In [1]:
# import relevant libraries
import pandas as pd
from itertools import islice
import itertools, csv, requests, json, ast, time
from pathlib import Path

## Step 1: Pre-Processing
This step cannot be done by this jupyter notebook. But in the future a server will be setup that runs alphafold3 and foldseek.

When running foldseek easy-search command, make sure to format the output using the following option:

`--format-output query,target,qstart,qend,tstart,tend,alntmscore,qtmscore,ttmscore,lddt,prob,evalue`

After completing this step the next step will require:
1. A name for the unknown protein
2. The foldseek output table
3. Positional information of the DUF you are interested in

In [2]:
# Enter the following information
# A name for the query protein containing domain of unknown function
query_protein = 'A0A015LDT9'
# Amino acid start and end location for the DUF in the query protein
start = 70
end = 290
# Path to the foldseek output file, include the filename as well, e.g. C:/Users/jodhs/Documents/EDGe_Lab/foldseek_file
foldseek_path = 'C:/Users/jodhs/Documents/EDGe_Lab/sample_foldseek_output.txt'

## Step 2: Filtering and Labeling 

This step will produce the following files:
1. A smaller version of the foldseek file with targets filered out using the positional information of the DUF
2. A json file containing information extracted from Uniprot about the domain description, positional information and weight of the matched proteins. (required for producing file 3.)
3. A labeled version of the foldseek file with each target getting a list of functional descriptions along with weights.  

In [3]:
def overlap(q1,q2,d1,d2):
    # check if matched region overlaps with DUF region
    # we know q2 > q1 and d2 > d1
    if q1 > d2 or d1 > q2:
        return False
    else:
        return True
        
def read_raw_foldseek_with_filter(foldseek_table, start, end):
    file_path = Path(foldseek_table).parent
    file_name = Path(foldseek_table).stem
    final_table = file_path / ('filtered_' + file_name)
    with open(foldseek_table, 'r') as infile, open(final_table, 'w', newline='') as outfile:
        reader = csv.reader(infile, delimiter='\t')
        writer = csv.writer(outfile)
        header = ['query','target','qstart','qend','tstart','tend','alntmscore','qtmscore','ttmscore',
                  'lddt','prob','evalue']
        writer.writerow(header)
        for row in reader:
            query = row[0]
            target = row[1].split('-')[1]
            qstart = int(row[2])
            qend = int(row[3])
            #drop rows containing self targets and drop matches that dont overlap with DUF in query
            if query != target and overlap(qstart, qend, start, end):
                row[0] = query
                row[1] = target
                # row.append(domain)
                # row.append(index)
                writer.writerow(row)
    return final_table
                
def manage_exception(data):
    matchID = data['extraAttributes']['uniParcId']
    base_url = 'https://rest.uniprot.org/uniparc/{}.json'.format(matchID)
    response = requests.get(base_url)
    data = response.json()
    function = False
    family_dict = []
    for database in data['uniParcCrossReferences']:
        if 'proteinName' in database.keys():
            function = database['proteinName']
            break
    if not function:
        return []
    seq_len = int(data['sequence']['length'])
    family_dict = [{"location": {"start": {"value": 1}, "end": {"value": seq_len}}, "description": function}]
    # print(json.dumps(family_dict))
    return json.dumps(family_dict)
    
def get_function_info(matchID):
    base_url = 'https://rest.uniprot.org/uniprotkb/{}.json?fields=ft_domain'.format(matchID)
    response = requests.get(base_url)
    # response = requests.get(base_url, headers=headers, params=params)
    if not response.ok:
        # response.raise_for_status()
        # print("bad response")
        family_dict = []
        return family_dict
        # sys.exit()
    
    data = response.json()

    if 'features' in data.keys() and (len(json.dumps(data['features'])) > 2):
        family_dict = json.dumps(data['features'])
        # print(family_dict)
    else:
        family_dict = manage_exception(data)

    return family_dict

# get information for each query and dump into json file
def make_info_json(raw_table):
    with open(raw_table, 'r') as infile:
        reader = csv.reader(infile)
        matchID_features = {}
        checkpoint = 0
        start_time = time.perf_counter()
        for row in reader:
            # key is query_target
            key = row[0]+"_"+row[1]
            matchID_features[key] = get_function_info(row[1])
            checkpoint+=1
            if checkpoint % 10 == 0:
                chk_time = time.perf_counter() -  start_time
                print("{} mins : {} matches labeled...".format(chk_time/60, checkpoint))
            # if checkpoint == (n_top*100 + 1):
                # break
        file_path = Path(raw_table).parent
        file_name = Path(raw_table).stem
        out_json = file_path / (file_name + '.json')
        with open(out_json, 'w') as outfile:
            json.dump(matchID_features, outfile, indent=2)
        return out_json

def get_matchIDs(input_table):
    match_IDs = []
    with open(input_table, 'r') as infile:
        f = csv.reader(infile)
        next(f)
        for row in f:
            match_IDs.append(row[1])
    return match_IDs



# function to add the overlap element to the family_dict by comparing the match's alignment range with each domain in the match
# using the information from family_dict.
def get_desc_and_overlap_info(json_list, m1, m2):
    # function returns a dictionary with {desc1: overlap, desc2: overlap ...} for a query_match
    # go over all the features and compaire start,end with matched residues
    info_dict = {}
    for i in range(len(json_list)):
        row = json_list[i]
        key = row['description']
        location = row['location']
        if location['start']['value'] and location['end']['value'] and key:
            try:
                d1 = int(location['start']['value'])
                d2 = int(location['end']['value'])
            except:
                continue
            if overlap(m1,m2,d1,d2):
                inputs = [m1,m2,d1,d2]
                inputs.sort()
                overlap_fraction = (inputs[2]-inputs[1]+1)/(d2-d1+1)
                info_dict[key] = overlap_fraction
        else:
            continue
    return info_dict

def append_desc_to_input_table(master_table, desc_file):
    file_path = Path(master_table).parent
    file_name = Path(master_table).stem
    out_table = file_path / ('labeled_' + file_name)
    with open(master_table, 'r') as intable, open(desc_file,'r') as infile, open(out_table, 'w', newline='') as outfile:
        reader_t = csv.reader(intable)
        reader = json.load(infile)
        writer = csv.writer(outfile)

        headers = next(reader_t)
        headers.append('description')
        writer.writerow(headers)

        for row_t in reader_t:
            query_match = row_t[0]+"_"+row_t[1]
            
            m1 = int(row_t[4])
            m2 = int(row_t[5])
            if reader[query_match]:
                json_list = json.loads(reader[query_match])
                info_dict = get_desc_and_overlap_info(json_list,m1,m2)
            else:
                info_dict = {}
            # print(info_dict)
            row_t.append(info_dict)
            writer.writerow(row_t)
        return out_table

filtered_table = read_raw_foldseek_with_filter(foldseek_path, start, end)
out_json = make_info_json(filtered_table)
labeled_table = append_desc_to_input_table(filtered_table, out_json)

0.18732328666665127 mins : 10 matches labeled...
0.37583771166585694 mins : 20 matches labeled...
0.5439872416657939 mins : 30 matches labeled...
0.7294582566663546 mins : 40 matches labeled...
0.8994476366662032 mins : 50 matches labeled...
1.1509965499998847 mins : 60 matches labeled...
1.3503522949991749 mins : 70 matches labeled...
1.5430802499991843 mins : 80 matches labeled...
1.7291780216658177 mins : 90 matches labeled...
1.9303337549994466 mins : 100 matches labeled...
2.149183386666118 mins : 110 matches labeled...
2.3508283533330543 mins : 120 matches labeled...
2.5433613716663483 mins : 130 matches labeled...
2.7380110416658376 mins : 140 matches labeled...
2.920233204999628 mins : 150 matches labeled...
3.1353406716661993 mins : 160 matches labeled...
3.344407718332756 mins : 170 matches labeled...
3.5526819449994944 mins : 180 matches labeled...
3.7260566083326316 mins : 190 matches labeled...
3.9110206483329724 mins : 200 matches labeled...
4.135062024999448 mins : 210 m

## Step 3: Aggragating and Ranking Functions

This step will produce a list of functions ranked in a descending order for the given query protein.

In [4]:
def aggragate_functions(name, in_table):
    file_path = Path(in_table).parent
    out_table = file_path / (name + '_ranked_functions')
    with open(out_table, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        header = [f'Ranked Functions for {name}', 'score']
        writer.writerow(header)

        df = pd.read_csv(in_table)

        query_desc = {}
        # pull all description dicts to a list for that query
        desc_list = df['description'].to_list()
            #print(desc_list)
        for desc in desc_list:
            desc_dict = ast.literal_eval(desc)
            # if empty desc, just skip everything
            if not desc_dict:
                continue
            keys = desc_dict.keys()
            for key in keys:
                if key in query_desc.keys():
                    query_desc[key] += desc_dict[key]
                else:
                    query_desc[key] = desc_dict[key]
        sorted_query_desc = dict(sorted(query_desc.items(), key=lambda item: item[1], reverse=True))

        for function in sorted_query_desc.keys():
            writer.writerow([function, sorted_query_desc[function]])
            print(function, sorted_query_desc[function])

aggragate_functions(query_protein, labeled_table)

Uncharacterized protein 71.75604673310632
Lipopolysaccharide 3-alpha-galactosyltransferase (Fragment) 29.865546310252785
Lipopolysaccharide 1,3-galactosyltransferase 26.972192550926927
Lipopolysaccharide 3-alpha-galactosyltransferase 17.25864378858017
Glycosyltransferase family 8 protein 12.670877520922481
UDP-D-galactose:(Glucosyl)lipopolysaccharide-alpha-1,3-D-galactosyltransferase 8.109561982373
Uncharacterized protein (Fragment) 7.80284325461122
DUF1647 domain-containing protein 7.6519197911565096
Glycosyl transferase 4.176801371000126
Glycosyl transferase family 8 3.740527940386014
Nucleotide-diphospho-sugar transferase domain-containing protein 3.5418197084744447
Lipopolysaccharide 1,3-galactosyltransferase (Fragment) 3.474622941620507
Glycosyl transferase family 8 C-terminal domain-containing protein 3.146496815286624
DUF5672 domain-containing protein 1.9037965679736817
Lipopolysaccharide 1,2-glucosyltransferase 1.4077380952380953
Uncharacterized protein LOC106059889 0.997590361